# Lab 02 — Building AI Agents using GitHub Models

This lab teaches how to build a simple **AI Agent using GitHub Models**.

Students will learn:

- How to call GitHub-hosted AI models
- What an AI agent is
- How agents use **tools**
- How to implement a **tool‑using agent loop**

No Azure subscription required. Only a **GitHub token**.

---

## Agent Concept

Agent = **LLM + Tools + Reasoning + Loop**

Flow:

User → LLM → Decide tool → Execute tool → Return result → Final answer

## Step 1 — Install Required Libraries
Run this once.

In [1]:
!pip install openai python-dotenv

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\hetarra\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Step 2 — Create Environment Variable

Create a `.env` file in the same folder.

Example:

```
GITHUB_TOKEN=your_token_here
```

You can create a token from:

https://github.com/settings/tokens

## Step 3 — Connect to GitHub Models
GitHub exposes models through an **OpenAI compatible API endpoint**.

In [2]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    api_key=os.getenv("GITHUB_TOKEN"),
    base_url="https://models.inference.ai.azure.com"
)

print("Client initialized")

Client initialized


## Step 4 — First LLM Call
Let's send a simple prompt to the model.

In [3]:
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "You are a helpful AI tutor."},
        {"role": "user", "content": "Explain what an AI agent is in simple terms."}
    ]
)

print(response.choices[0].message.content)

Sure! An AI agent is like a smart helper or a virtual assistant that can make decisions and take actions to achieve specific goals. It uses artificial intelligence to understand its environment, think about what to do, and then act on it.

For example, if you have a navigation app, the app can suggest the best route to your destination. In this case, the app acts as an AI agent: it gathers information about traffic, thinks about the fastest way to get there, and gives you directions.

So in simple terms, an AI agent is a computer program that senses, thinks, and acts intelligently to get things done!


## Step 5 — Understanding Tools

Agents can call **external tools**.

Example tools:

- Weather API
- Database lookup
- Calculator
- Search

We'll simulate a weather API.

In [5]:
def get_weather(city):
    weather_data = {
        "Hyderabad": "35°C and sunny",
        "Seattle": "10°C and rainy",
        "London": "12°C cloudy",
        "Delhi": "30°C hot"
    }

    return weather_data.get(city, "Weather data not available")

## Step 6 — Define Tool Schema

LLMs need a **JSON schema** describing tools.

In [6]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get weather information for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "Name of the city"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

print("Tool schema ready")

Tool schema ready


## Step 7 — Ask the Model a Question

The model will decide if it needs to call a tool.

In [7]:
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "user", "content": "What is the weather in Hyderabad?"}
    ],
    tools=tools,
    tool_choice="auto"
)

response

ChatCompletion(id='chatcmpl-DJN43u7pwSELIpKivCIWuh0v095fG', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_d89goU02o28dKAk2UQfBdFpX', function=Function(arguments='{"city":"Hyderabad"}', name='get_weather'), type='function')]), content_filter_results={})], created=1773509263, model='gpt-4o-2024-11-20', object='chat.completion', service_tier=None, system_fingerprint='fp_af7f7349a4', usage=CompletionUsage(completion_tokens=16, prompt_tokens=57, total_tokens=73, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)), prompt_filter_results=[{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity':

## Step 8 — Check if Tool Was Called

In [8]:
tool_call = response.choices[0].message.tool_calls[0]

tool_call

ChatCompletionMessageFunctionToolCall(id='call_d89goU02o28dKAk2UQfBdFpX', function=Function(arguments='{"city":"Hyderabad"}', name='get_weather'), type='function')

## Step 9 — Execute the Tool

In [9]:
import json

city = json.loads(tool_call.function.arguments)["city"]

result = get_weather(city)

print("Tool result:", result)

Tool result: 35°C and sunny


## Step 10 — Send Tool Result Back to Model

In [11]:
messages = [
    {"role": "user", "content": "What is the weather in Hyderabad?"},
    response.choices[0].message,
    {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": result
    }
]

final_response = client.chat.completions.create(
    model="gpt-4o",
    messages=messages
)

print(final_response.choices[0].message.content)

The weather in Hyderabad is currently 35°C and sunny.


# 🎉 Congratulations

You just built a **working AI agent**.

The agent:

1. Receives a question
2. Decides to call a tool
3. Executes the tool
4. Uses the result to generate an answer

---

# Exercises for Students

### Exercise 1
Add more cities to the weather tool.

### Exercise 2
Create a new tool:

```
get_population(city)
```

### Exercise 3
Create a **calculator tool**.

### Exercise 4
Allow the agent to use **multiple tools**.

---

# Next Lab Ideas

Lab 3 — RAG Agent  
Lab 4 — Multi-Agent Systems  
Lab 5 — Autonomous Agents